In [1]:
# Ensure Homebrew binaries are in PATH
import os
os.environ['PATH'] = '/opt/homebrew/bin:' + os.environ.get('PATH', '')

In [2]:
# Install dependencies if needed
# !pip install langchain langchain-experimental langchain-chroma unstructured pydantic
import os
from textbook_loading_text_only import (
    load_book,
    clean_and_categorize_elements,
    summarize_elements,
    store_in_chromadb,
)

In [3]:
pdf_file = '../../data/Cat_Owners_Home_Veterinary_Handbook_Trimed.pdf'
chroma_persist_dir = '../../chroma/TO_Cat_Owners_Home_Veterinary_Handbook_Trimed/'

# Make sure the data directory exists
assert os.path.exists('../../data'), "Error: '../../data' directory not found."
assert os.path.exists(pdf_file), f"Error: PDF file not found at {pdf_file}."

#Small sample testing (works)
# pdf_file = '../../data/MediumExample_Ears_17Pgs.pdf'
# chroma_persist_dir = '../../chroma/TO_MediumExample_Ears_17Pgs/'

# # Make sure the data directory exists
# assert os.path.exists('../../data'), "Error: '../../data' directory not found."
# assert os.path.exists(pdf_file), f"Error: PDF file not found at {pdf_file}."

In [4]:
print("📝 Unstructuring textbooks, filtering junks, semantic chunking...")
raw_pdf_elements = load_book(pdf_file)
print("🎉 1.process_pdf_with_semantic_chunking complete.")

📝 Unstructuring textbooks, filtering junks, semantic chunking...


The `max_size` parameter is deprecated and will be removed in v4.26. Please specify in `size['longest_edge'] instead`.


🎉 1.process_pdf_with_semantic_chunking complete.


In [5]:
# Clean and categorize (text and tables only)
texts, tables = clean_and_categorize_elements(raw_pdf_elements, min_meaningful_text_length=75)

In [6]:
# Check how many texts and tables we have
print(f"Number of text elements: {len(texts)}")
print(f"Number of table elements: {len(tables)}")
print(f"Total elements to summarize: {len(texts) + len(tables)}")

Number of text elements: 1261
Number of table elements: 25
Total elements to summarize: 1286


In [7]:
# Summarize text and tables
text_summaries, table_summaries = summarize_elements(texts, tables, raw_pdf_elements)

Performing semantic chunking on texts...


/Users/mas/Desktop/LLM_Veterinary_AI/pawsitive_v1_workflow/textbook_to_db/textbook_loading_text_only.py:168: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(model_name="Qwen/Qwen3-Embedding-0.6B")


Created 621 semantic chunks from 1261 text elements
Summarizing 621 chunks with Ollama (concurrency=8)...
Summarizing 25 tables with Ollama (concurrency=8)...
Summarizing 25 tables with Ollama (concurrency=8)...
Texts and Tables Summary Done!
Texts and Tables Summary Done!


In [8]:
import subprocess
import re
from datetime import datetime

print(f"⏰ Check Time: {datetime.now().strftime('%H:%M:%S')}\n")

# Check Ollama runner process
result = subprocess.run(
    ["ps", "aux"], 
    capture_output=True, 
    text=True
)

for line in result.stdout.split('\n'):
    if 'ollama runner' in line.lower():
        parts = line.split()
        cpu = parts[2]
        mem = parts[3]
        time = parts[9]
        print(f"🤖 Ollama Runner:")
        print(f"   CPU: {cpu}%  Memory: {mem}%  Runtime: {time}")
        
        # Check if it's actually working
        cpu_float = float(cpu)
        if cpu_float > 5:
            print(f"   ✅ ACTIVELY PROCESSING (CPU > 5%)")
        elif cpu_float > 0.1:
            print(f"   ⚠️  LOW ACTIVITY (might be waiting on I/O)")
        else:
            print(f"   ❌ STUCK! (CPU = 0%)")
        break
else:
    print("❌ Ollama runner not found - might have crashed!")

print("\n" + "="*50)
print("💡 TIP: If CPU < 1%, the process might be stuck!")

⏰ Check Time: 22:34:56

🤖 Ollama Runner:
   CPU: 10.7%  Memory: 12.3%  Runtime: 10:48.92
   ✅ ACTIVELY PROCESSING (CPU > 5%)

💡 TIP: If CPU < 1%, the process might be stuck!


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [9]:
# Check if any summaries have been generated (without interrupting)
import os
import sys

# Get reference to the running cell's variables
if 'text_summaries' in dir():
    print(f"✅ text_summaries exists: {len(text_summaries)} summaries generated")
else:
    print("⏳ text_summaries not yet created - still in semantic chunking phase")
    
if 'table_summaries' in dir():
    print(f"✅ table_summaries exists: {len(table_summaries)} summaries")
else:
    print("⏳ table_summaries not yet created")

✅ text_summaries exists: 621 summaries generated
✅ table_summaries exists: 25 summaries


In [10]:
import importlib
import gc
import torch

# Clear GPU memory first
if torch.backends.mps.is_available():
    torch.mps.empty_cache()
    print("✅ Cleared MPS (GPU) cache")
gc.collect()
print("✅ Ran garbage collection")

# Reload the module to get updated code
import textbook_loading_text_only
importlib.reload(textbook_loading_text_only)
from textbook_loading_text_only import store_in_chromadb

print("✅ Module reloaded with new memory-optimized code")
print("\n💡 Now run the next cell to store in ChromaDB")

✅ Cleared MPS (GPU) cache
✅ Ran garbage collection
✅ Module reloaded with new memory-optimized code

💡 Now run the next cell to store in ChromaDB


## 🔄 Reload Module & Clear Memory

Run this before retrying the ChromaDB storage to get the updated code with GPU memory management

## 🔍 Monitoring Progress

Run this cell periodically to check if summarization is actually progressing (not stuck)

In [11]:
# Store in ChromaDB
retriever = store_in_chromadb(
    text_summaries, texts, table_summaries, tables,
    persist_directory=chroma_persist_dir
)

🧹 Cleared GPU memory before embedding
Adding 621 text summaries in batches of 5...
Adding 621 text summaries in batches of 5...
  Processed batch 1/125
  Processed batch 1/125
  Processed batch 2/125
  Processed batch 2/125
  Processed batch 3/125
  Processed batch 3/125
  Processed batch 4/125
  Processed batch 4/125
  Processed batch 5/125
  Processed batch 5/125
  Processed batch 6/125
  Processed batch 6/125
  Processed batch 7/125
  Processed batch 7/125
  Processed batch 8/125
  Processed batch 8/125
  Processed batch 9/125
  Processed batch 9/125
  Processed batch 10/125
  Processed batch 10/125
  Processed batch 11/125
  Processed batch 11/125
  Processed batch 12/125
  Processed batch 12/125
  Processed batch 13/125
  Processed batch 13/125
  Processed batch 14/125
  Processed batch 14/125
  Processed batch 15/125
  Processed batch 15/125
  Processed batch 16/125
  Processed batch 16/125
  Processed batch 17/125
  Processed batch 17/125
  Processed batch 18/125
  Processed bat

In [12]:
# System sound, when done
sound_file = "/System/Library/Sounds/Glass.aiff"
os.system(f"afplay '{sound_file}'")

0

# Inspecting Retrieved Docs

In [13]:
query = "My cat has being scratching its ear too often. There are some dark greasy thing in it. It scratch its ear so often and so hard that I see wounds and blood in it. What should I do?"
results = retriever.retrieve_multi_modal(query, k=5)

In [14]:
# Display retrieved text chunks
print('-'*40, "Retrieved Text Chunks (first 300 chars)", '-'*40)
for res in results:
    if res["modality"] == "text":
        doc_id = res["original_metadata"].get("doc_id")
        original_text = None
        if doc_id and hasattr(retriever, "text_docstore"):
            doc = retriever.text_docstore._collection.get(ids=[doc_id], include=["documents"])
            if doc and doc.get("documents") and doc["documents"][0]:
                original_text = doc["documents"][0]
        if not original_text:
            original_text = res["summary"]
        text_display = original_text[:300] + ("..." if len(original_text) > 300 else "")
        print(text_display)
        print('-'*20)
    elif res["modality"] == "table":
        doc_id = res["original_metadata"].get("doc_id")
        print(f"[TABLE] Summary: {res['summary'][:200]}...")
        print('-'*20)

---------------------------------------- Retrieved Text Chunks (first 300 chars) ----------------------------------------
Here's a concise summary:

**External Ear Problems in Cats:**

* Signs: discharge, head shaking, ear scratching, tenderness
* Severe scratching can lead to abraded skin and infection (abscess)
* Identify and treat underlying cause of itching/scratching for effective treatment.
--------------------
**Summary:**

* Clean cat bites or lacerations with Betadine or chlorhexidine solution, avoiding eyes.
* Apply topical antibiotic ointment after cleaning.
* Distract cat to prevent rubbing or licking off ointment.
* Consider antibiotics (consult vet) to prevent abscesses.
* Large lacerations or thos...
--------------------
**Hematoma in Cats:**

* Caused by trauma, violent head shaking, or scratching
* Often linked to underlying conditions like ear mites or infections
* Treatment:
	+ Express blood from hematoma with vet's assistance
	+ Surgery (skin window and drainage) fo